# Voting Ensemble: Hard vs. Soft Voting

This notebook explores voting ensembles, which combine predictions from multiple classifiers using either hard voting (majority vote) or soft voting (average probabilities).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Create Hard Voting Ensemble

In [ ]:
# Hard voting: majority vote
hard_voting = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svm', SVC(kernel='rbf', random_state=42))
    ],
    voting='hard'
)

hard_voting.fit(X_train, y_train)
y_pred_hard = hard_voting.predict(X_test)
acc_hard = accuracy_score(y_test, y_pred_hard)

print(f"Hard Voting Ensemble Accuracy: {acc_hard:.4f}")

## 3. Create Soft Voting Ensemble

In [ ]:
# Soft voting: average probabilities
soft_voting = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42, probability=True)),
        ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svm', SVC(kernel='rbf', random_state=42, probability=True))
    ],
    voting='soft'
)

soft_voting.fit(X_train, y_train)
y_pred_soft = soft_voting.predict(X_test)
acc_soft = accuracy_score(y_test, y_pred_soft)

print(f"Soft Voting Ensemble Accuracy: {acc_soft:.4f}")

## 4. Compare with Individual Classifiers

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42)
}

results = {}
print("\n=== INDIVIDUAL CLASSIFIER ACCURACIES ===")
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:20s}: {acc:.4f}")

print("\n=== ENSEMBLE ACCURACIES ===")
print(f"Hard Voting:           {acc_hard:.4f}")
print(f"Soft Voting:           {acc_soft:.4f}")

best_base = max(results.values())
print(f"\nBest base classifier:  {best_base:.4f}")
print(f"Hard voting improvement:  {acc_hard - best_base:+.4f}")
print(f"Soft voting improvement:  {acc_soft - best_base:+.4f}")

## 5. Classification Reports

In [ ]:
print("\nHard Voting Classification Report:")
print(classification_report(y_test, y_pred_hard, target_names=target_names))

print("\nSoft Voting Classification Report:")
print(classification_report(y_test, y_pred_soft, target_names=target_names))

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_hard = confusion_matrix(y_test, y_pred_hard)
sns.heatmap(cm_hard, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, 
            yticklabels=target_names, ax=axes[0])
axes[0].set_title(f'Hard Voting (Acc: {acc_hard:.4f})')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

cm_soft = confusion_matrix(y_test, y_pred_soft)
sns.heatmap(cm_soft, annot=True, fmt='d', cmap='Greens', xticklabels=target_names,
            yticklabels=target_names, ax=axes[1])
axes[1].set_title(f'Soft Voting (Acc: {acc_soft:.4f})')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## 7. Hard vs. Soft Voting Comparison

In [ ]:
comparison = {
    'Aspect': ['Decision Rule', 'When to use', 'Data requirement', 'Robustness'],
    'Hard Voting': [
        'Majority class prediction',
        'All classifiers equally diverse',
        'Less sensitive to probability calibration',
        'Simple but can miss nuances'
    ],
    'Soft Voting': [
        'Average predicted probabilities',
        'Classifiers with calibrated probabilities',
        'Requires probability estimates',
        'More nuanced, weighs confidence'
    ]
}

df_comparison = pd.DataFrame(comparison)
print(df_comparison.to_string(index=False))

## 8. Key Insights

- **Hard voting:** Simple majority - good when classifiers equally trained
- **Soft voting:** Weighted by confidence - better with calibrated probabilities
- **Ensemble diversity:** Combining uncorrelated classifiers maximizes benefit
- **Typical improvement:** 1-3% over best base classifier